# 02-13b nnU-Net ICH teacher external data - improved

This notebook keeps the original 02_13 teacher pipeline, but adds the improvements from `improve_02_13_02_14_pipeline_plan.md`:

- explicit label-value QC before converting `label > 0`;
- configurable ICH label mapping per external dataset;
- teacher probability-map export in addition to binary masks;
- QC selection that includes low Dice, high Dice, tiny prediction, and large prediction cases;
- multi-window teacher visualization for external test cases.

The original 02_13 notebook is left untouched. This branch writes to `outputs_02_13b_nnunet_ich_teacher_external_improved`.


In [1]:
from pathlib import Path
import os
import sys
import json
import shutil
import time
import gc

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
NNUNET_REPO = PROJECT_ROOT / 'nnUNet-master'

SEG_CQ500_ROOT = PROJECT_ROOT / 'Seg-CQ500' / 'Seg-CQ500' / 'data' / 'volumes'
INSTANCE_ROOT = PROJECT_ROOT / 'Instance2022'
INSTANCE_TRAIN_DATA = INSTANCE_ROOT / 'train_2' / 'data'
INSTANCE_TRAIN_LABEL = INSTANCE_ROOT / 'train_2' / 'label'

OUTPUT_ROOT = PROJECT_ROOT / 'outputs_02_13b_nnunet_ich_teacher_external_improved'
NNUNET_BASE = OUTPUT_ROOT / 'nnunet_workdir'
NNUNET_RAW = NNUNET_BASE / 'nnUNet_raw'
NNUNET_PREPROCESSED = NNUNET_BASE / 'nnUNet_preprocessed'
NNUNET_RESULTS = NNUNET_BASE / 'nnUNet_results'

DATASET_ID = 123
DATASET_NAME = f'Dataset{DATASET_ID:03d}_ExternalICHTeacherImproved'
DATASET_DIR = NNUNET_RAW / DATASET_NAME

for p in [OUTPUT_ROOT, NNUNET_RAW, NNUNET_PREPROCESSED, NNUNET_RESULTS]:
    p.mkdir(parents=True, exist_ok=True)

os.environ['nnUNet_raw'] = str(NNUNET_RAW)
os.environ['nnUNet_preprocessed'] = str(NNUNET_PREPROCESSED)
os.environ['nnUNet_results'] = str(NNUNET_RESULTS)

# Safe defaults for Windows/Jupyter. Planning temporarily switches DA workers to 1.
os.environ['nnUNet_n_proc_DA'] = '0'
os.environ['nnUNet_compile'] = 'False'

if str(NNUNET_REPO) not in sys.path:
    sys.path.insert(0, str(NNUNET_REPO))

assert NNUNET_REPO.exists(), f'Missing nnU-Net repo: {NNUNET_REPO}'
assert SEG_CQ500_ROOT.exists(), f'Missing Seg-CQ500 volumes dir: {SEG_CQ500_ROOT}'
assert INSTANCE_TRAIN_DATA.exists(), f'Missing Instance2022 data dir: {INSTANCE_TRAIN_DATA}'
assert INSTANCE_TRAIN_LABEL.exists(), f'Missing Instance2022 label dir: {INSTANCE_TRAIN_LABEL}'

print('Project root          :', PROJECT_ROOT)
print('nnU-Net repo          :', NNUNET_REPO)
print('Seg-CQ500 volumes     :', SEG_CQ500_ROOT)
print('Instance2022 data     :', INSTANCE_TRAIN_DATA)
print('Instance2022 label    :', INSTANCE_TRAIN_LABEL)
print('nnUNet_raw           :', os.environ['nnUNet_raw'])
print('nnUNet_preprocessed  :', os.environ['nnUNet_preprocessed'])
print('nnUNet_results       :', os.environ['nnUNet_results'])
print('Dataset              :', DATASET_NAME)
print('nnUNet_n_proc_DA     :', os.environ['nnUNet_n_proc_DA'])
print('nnUNet_compile       :', os.environ['nnUNet_compile'])


Project root          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao
nnU-Net repo          : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master
Seg-CQ500 volumes     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Seg-CQ500\data\volumes
Instance2022 data     : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Instance2022\train_2\data
Instance2022 label    : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Instance2022\train_2\label
nnUNet_raw           : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_raw
nnUNet_preprocessed  : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_preprocessed
nnUNet_results       : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_results
Dataset              : Dataset123_ExternalICHTeacherImproved
nnUNet_n_proc_DA     : 0
nnUNet_compile       : False


## 0. Dependency check

Cell này cài dependency thiếu nếu môi trường local chưa đủ. `SimpleITK` là bắt buộc vì nnU-Net đang dùng `SimpleITKIO` cho NIfTI.

In [2]:
INSTALL_NNUNET = False
AUTO_INSTALL_MISSING_DEPS = True

def ensure_import(import_name, pip_name=None):
    import importlib
    import subprocess
    pip_name = pip_name or import_name
    try:
        return importlib.import_module(import_name)
    except ModuleNotFoundError:
        if not AUTO_INSTALL_MISSING_DEPS:
            raise
        cmd = [sys.executable, '-m', 'pip', 'install', pip_name]
        print('Missing', import_name, '-> running:', ' '.join(cmd))
        subprocess.check_call(cmd)
        return importlib.import_module(import_name)

if INSTALL_NNUNET:
    import subprocess
    cmd = [sys.executable, '-m', 'pip', 'install', '-e', str(NNUNET_REPO)]
    print('Running:', ' '.join(cmd))
    subprocess.check_call(cmd)

ensure_import('SimpleITK')
ensure_import('nibabel')
ensure_import('scipy')
ensure_import('skimage', 'scikit-image')
ensure_import('batchgenerators')
ensure_import('batchgeneratorsv2')
ensure_import('acvl_utils', 'acvl-utils')
ensure_import('dynamic_network_architectures', 'dynamic-network-architectures')
ensure_import('blosc2')
ensure_import('yacs')

import nnunetv2
print('nnunetv2 import OK:', nnunetv2.__file__)


nnunetv2 import OK: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\nnUNet-master\nnunetv2\__init__.py


## 0b. Label mapping policy

The default remains conservative: if a dataset-specific ICH label list is unknown, the notebook uses `label > 0`.

If QC shows that a dataset contains non-ICH foreground classes, set the matching entry in `ICH_LABEL_VALUES_BY_DATASET` to the exact hemorrhage labels, for example `[1]`.


In [3]:
ICH_LABEL_VALUES_BY_DATASET = {
    'Seg-CQ500': None,       # None means fallback to label > 0.
    'Instance2022': None,    # Change to [1] etc. if label QC proves a stricter mapping.
}

TEACHER_PROB_THRESHOLD = 0.5

def label_to_ich_mask(label_arr, dataset_name):
    arr = np.asarray(label_arr)
    label_values = ICH_LABEL_VALUES_BY_DATASET.get(str(dataset_name), None)
    if label_values is None:
        return (arr > 0).astype(np.uint8)
    return np.isin(arr, list(label_values)).astype(np.uint8)

def summarize_label_values(label_arr, max_values=40):
    values, counts = np.unique(label_arr, return_counts=True)
    rows = []
    for value, count in zip(values[:max_values], counts[:max_values]):
        rows.append({'label_value': str(value), 'voxels': int(count)})
    if len(values) > max_values:
        rows.append({'label_value': f'... {len(values) - max_values} more', 'voxels': int(counts[max_values:].sum())})
    return rows

print('ICH label policy:', ICH_LABEL_VALUES_BY_DATASET)
print('Teacher probability threshold:', TEACHER_PROB_THRESHOLD)


ICH label policy: {'Seg-CQ500': None, 'Instance2022': None}
Teacher probability threshold: 0.5


## 1. Build manifest external ICH

Label policy trong notebook này:

```text
0 = background
1 = ICH / hemorrhage-like lesion
```

Với `Instance2022`, label được ép binary bằng `label > 0`. Cần QC sau khi convert để chắc label đúng loại tổn thương mình muốn dùng làm ICH teacher.

In [4]:
USE_SEG_CQ500 = True
USE_INSTANCE2022 = True

rows = []

if USE_SEG_CQ500:
    for case_dir in sorted(SEG_CQ500_ROOT.iterdir()):
        if not case_dir.is_dir():
            continue
        image_path = case_dir / 'CT.nii'
        label_path = case_dir / 'ICH_mask.nii.gz'
        if image_path.exists() and label_path.exists():
            safe_id = ''.join(ch if ch.isalnum() else '_' for ch in case_dir.name).strip('_')
            rows.append({
                'dataset': 'Seg-CQ500',
                'case_id': f'segcq500_{safe_id}',
                'image_path': str(image_path),
                'label_path': str(label_path),
                'label_policy': 'ICH_mask_binary',
            })

if USE_INSTANCE2022:
    for image_path in sorted(INSTANCE_TRAIN_DATA.glob('*.nii.gz')):
        label_path = INSTANCE_TRAIN_LABEL / image_path.name
        if image_path.exists() and label_path.exists():
            stem = image_path.name.replace('.nii.gz', '')
            rows.append({
                'dataset': 'Instance2022',
                'case_id': f'inst2022_{stem}',
                'image_path': str(image_path),
                'label_path': str(label_path),
                'label_policy': 'label_gt_0_binary',
            })

manifest_df = pd.DataFrame(rows)
assert len(manifest_df) > 0, 'No external ICH cases found.'
assert manifest_df['case_id'].is_unique, 'case_id must be unique.'

display(manifest_df.groupby('dataset').size().rename('cases').reset_index())
display(manifest_df.head())
manifest_df.to_csv(OUTPUT_ROOT / '02_13_external_ich_manifest_raw.csv', index=False)


,dataset,cases
0,Instance2022,100
1,Seg-CQ500,51


,dataset,case_id,image_path,label_path,label_policy
0,Seg-CQ500,segcq500_CQ500_CT_10,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,ICH_mask_binary
1,Seg-CQ500,segcq500_CQ500_CT_108,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,ICH_mask_binary
2,Seg-CQ500,segcq500_CQ500_CT_113,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,ICH_mask_binary
3,Seg-CQ500,segcq500_CQ500_CT_121,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,ICH_mask_binary
4,Seg-CQ500,segcq500_CQ500_CT_125,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\Seg-CQ500\Se...,ICH_mask_binary


## 2. Inspect label values nhanh

Cell này đọc vài label để xem label có giá trị gì. Với teacher ICH, notebook sẽ convert mọi voxel `> 0` thành class 1.

In [5]:
import SimpleITK as sitk
import nibabel as nib

INSPECT_LABELS = True
MAX_INSPECT_PER_DATASET = 3
RUN_FULL_LABEL_QC = True
ITK_ORTHO_ERROR = 'orthonormal direction cosines'

def read_image(path: Path, dtype=None):
    path = Path(path)
    try:
        return sitk.ReadImage(str(path))
    except RuntimeError as e:
        if ITK_ORTHO_ERROR not in str(e):
            raise
        nii = nib.load(str(path))
        data = np.asanyarray(nii.dataobj)
        if data.ndim == 4 and data.shape[-1] == 1:
            data = data[..., 0]
        if data.ndim != 3:
            raise ValueError(f'Expected 3D NIfTI, got shape {data.shape}: {path}')
        if dtype is not None:
            data = data.astype(dtype, copy=False)
        img = sitk.GetImageFromArray(np.asarray(data).transpose(2, 1, 0))
        img.SetSpacing(tuple(float(abs(x)) for x in nii.header.get_zooms()[:3]))
        img.SetOrigin(tuple(float(x) for x in nii.affine[:3, 3]))
        img.SetDirection(tuple(np.eye(3).ravel()))
        img.SetMetaData('reader_fallback', 'nibabel_sanitized_direction')
        return img

if INSPECT_LABELS:
    inspect_rows = []
    for dataset_name, group in manifest_df.groupby('dataset'):
        for row in group.head(MAX_INSPECT_PER_DATASET).itertuples(index=False):
            seg = read_image(Path(row.label_path))
            arr = sitk.GetArrayFromImage(seg)
            ich_mask = label_to_ich_mask(arr, dataset_name)
            values, counts = np.unique(arr, return_counts=True)
            inspect_rows.append({
                'dataset': dataset_name,
                'case_id': row.case_id,
                'shape_zyx': tuple(arr.shape),
                'spacing_xyz': tuple(float(x) for x in seg.GetSpacing()),
                'reader': seg.GetMetaData('reader_fallback') if seg.HasMetaDataKey('reader_fallback') else 'SimpleITK',
                'unique_values': values[:20].tolist(),
                'nonzero_voxels': int((arr > 0).sum()),
                'mapped_ich_voxels': int(ich_mask.sum()),
                'mapping_policy': ICH_LABEL_VALUES_BY_DATASET.get(str(dataset_name), None) or 'label_gt_0',
            })
    display(pd.DataFrame(inspect_rows))
else:
    print('Skipped label inspection.')

if RUN_FULL_LABEL_QC:
    label_count_rows = []
    for row in manifest_df.itertuples(index=False):
        seg = read_image(Path(row.label_path))
        arr = sitk.GetArrayFromImage(seg)
        values, counts = np.unique(arr, return_counts=True)
        for value, count in zip(values, counts):
            label_count_rows.append({
                'dataset': row.dataset,
                'case_id': row.case_id,
                'label_value': str(value),
                'voxels': int(count),
            })
    label_counts_df = pd.DataFrame(label_count_rows)
    label_counts_path = OUTPUT_ROOT / '02_13b_external_label_value_counts.csv'
    label_counts_df.to_csv(label_counts_path, index=False)
    display(label_counts_df.groupby(['dataset', 'label_value'])['voxels'].agg(['count', 'sum']).reset_index())
    print('Saved label counts:', label_counts_path)
else:
    print('Skipped full label-value QC.')


,dataset,case_id,shape_zyx,spacing_xyz,reader,unique_values,nonzero_voxels,mapped_ich_voxels,mapping_policy
0,Instance2022,inst2022_001,"(29, 512, 512)","(0.44600000977516174, 0.44600000977516174, 5.0)",SimpleITK,"[0, 1]",198,198,label_gt_0
1,Instance2022,inst2022_002,"(28, 512, 512)","(0.48828125, 0.48828125, 5.0)",SimpleITK,"[0, 1]",5624,5624,label_gt_0
2,Instance2022,inst2022_003,"(34, 512, 512)","(0.48828125, 0.48828125, 5.0)",SimpleITK,"[0, 1]",2378,2378,label_gt_0
3,Seg-CQ500,segcq500_CQ500_CT_10,"(224, 512, 512)","(0.4648439884185791, 0.4648439586162567, 0.631...",nibabel_sanitized_direction,"[0, 1]",23118,23118,label_gt_0
4,Seg-CQ500,segcq500_CQ500_CT_108,"(256, 512, 512)","(0.4960939884185791, 0.49609386920928955, 0.68...",nibabel_sanitized_direction,"[0, 1]",2930417,2930417,label_gt_0
5,Seg-CQ500,segcq500_CQ500_CT_113,"(272, 512, 512)","(0.46875, 0.4687500596046448, 0.6303173303604126)",nibabel_sanitized_direction,"[0, 1]",33167,33167,label_gt_0


,dataset,label_value,count,sum
0,Instance2022,0,100,778835310
1,Instance2022,1,100,2615954
2,Seg-CQ500,0,51,2919824534
3,Seg-CQ500,1,48,9314289
4,Seg-CQ500,2,3,320377


Saved label counts: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\02_13b_external_label_value_counts.csv


## 3. Split external ICH dataset

Split được tạo riêng theo từng dataset để cả Seg-CQ500 và Instance2022 đều có mặt trong train/val/test.

In [6]:
SEED = 42
VAL_FRACTION = 0.15
TEST_FRACTION = 0.15

rng = np.random.default_rng(SEED)
split_values = []

for dataset_name, group in manifest_df.groupby('dataset', sort=True):
    idx = group.index.to_numpy(copy=True)
    rng.shuffle(idx)
    n = len(idx)
    n_test = max(1, int(round(n * TEST_FRACTION)))
    n_val = max(1, int(round(n * VAL_FRACTION)))
    test_idx = set(idx[:n_test])
    val_idx = set(idx[n_test:n_test + n_val])
    for i in group.index:
        if i in test_idx:
            split_values.append((i, 'test'))
        elif i in val_idx:
            split_values.append((i, 'val'))
        else:
            split_values.append((i, 'train'))

split_series = pd.Series({i: s for i, s in split_values})
manifest_df['split'] = manifest_df.index.map(split_series)

display(manifest_df.groupby(['dataset', 'split']).size().rename('cases').reset_index())
display(manifest_df.groupby('split').size().rename('cases').reset_index())
manifest_df.to_csv(OUTPUT_ROOT / '02_13_external_ich_manifest_split.csv', index=False)


,dataset,split,cases
0,Instance2022,test,15
1,Instance2022,train,70
2,Instance2022,val,15
3,Seg-CQ500,test,8
4,Seg-CQ500,train,35
5,Seg-CQ500,val,8


,split,cases
0,test,23
1,train,105
2,val,23


## 4. Convert sang nnU-Net raw dataset

nnU-Net yêu cầu image và label có cùng file ending. Notebook này ghi lại tất cả ảnh/label thành `.nii.gz` bằng SimpleITK.

Cấu trúc output:

```text
nnUNet_raw/Dataset113_ExternalICHTeacher/
  imagesTr/*_0000.nii.gz
  labelsTr/*.nii.gz
  imagesTs/*_0000.nii.gz
  labelsTs/*.nii.gz
  dataset.json
```

In [7]:
OVERWRITE_RAW_DATASET = True

import SimpleITK as sitk
import nibabel as nib

ITK_ORTHO_ERROR = 'orthonormal direction cosines'

def read_image(path: Path, dtype=None):
    path = Path(path)
    try:
        return sitk.ReadImage(str(path))
    except RuntimeError as e:
        if ITK_ORTHO_ERROR not in str(e):
            raise
        nii = nib.load(str(path))
        data = np.asanyarray(nii.dataobj)
        if data.ndim == 4 and data.shape[-1] == 1:
            data = data[..., 0]
        if data.ndim != 3:
            raise ValueError(f'Expected 3D NIfTI, got shape {data.shape}: {path}')
        if dtype is not None:
            data = data.astype(dtype, copy=False)
        img = sitk.GetImageFromArray(np.asarray(data).transpose(2, 1, 0))
        img.SetSpacing(tuple(float(abs(x)) for x in nii.header.get_zooms()[:3]))
        img.SetOrigin(tuple(float(x) for x in nii.affine[:3, 3]))
        img.SetDirection(tuple(np.eye(3).ravel()))
        img.SetMetaData('reader_fallback', 'nibabel_sanitized_direction')
        return img

def write_case_niigz(image_src_path: Path, label_src_path: Path, img_dst: Path, seg_dst: Path, dataset_name: str):
    img = read_image(image_src_path)
    seg = read_image(label_src_path)
    arr = sitk.GetArrayFromImage(seg)
    mask = label_to_ich_mask(arr, dataset_name)
    out = sitk.GetImageFromArray(mask.astype(np.uint8))
    if img.GetSize() != out.GetSize():
        raise ValueError(
            f'Image/label size mismatch: {image_src_path} {img.GetSize()} vs '
            f'{label_src_path} {out.GetSize()}'
        )
    out.CopyInformation(img)
    sitk.WriteImage(img, str(img_dst))
    sitk.WriteImage(out, str(seg_dst))
    return int(mask.sum())

if DATASET_DIR.exists() and OVERWRITE_RAW_DATASET:
    shutil.rmtree(DATASET_DIR)

imagesTr = DATASET_DIR / 'imagesTr'
labelsTr = DATASET_DIR / 'labelsTr'
imagesTs = DATASET_DIR / 'imagesTs'
labelsTs = DATASET_DIR / 'labelsTs'
for p in [imagesTr, labelsTr, imagesTs, labelsTs]:
    p.mkdir(parents=True, exist_ok=True)

converted_rows = []
for row in manifest_df.itertuples(index=False):
    case_id = str(row.case_id)
    if row.split == 'test':
        img_dst = imagesTs / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTs / f'{case_id}.nii.gz'
    else:
        img_dst = imagesTr / f'{case_id}_0000.nii.gz'
        seg_dst = labelsTr / f'{case_id}.nii.gz'
    ich_voxels = write_case_niigz(Path(row.image_path), Path(row.label_path), img_dst, seg_dst, row.dataset)
    converted_rows.append({
        'dataset': row.dataset,
        'case_id': case_id,
        'split': row.split,
        'nnunet_image': str(img_dst),
        'nnunet_label': str(seg_dst),
        'ich_positive_voxels': ich_voxels,
        'label_mapping': ICH_LABEL_VALUES_BY_DATASET.get(str(row.dataset), None) or 'label_gt_0',
    })

dataset_json = {
    'channel_names': {'0': 'CT'},
    'labels': {'background': 0, 'ICH': 1},
    'numTraining': int((manifest_df['split'] != 'test').sum()),
    'file_ending': '.nii.gz',
    'overwrite_image_reader_writer': 'SimpleITKIO',
}
with open(DATASET_DIR / 'dataset.json', 'w', encoding='utf-8') as f:
    json.dump(dataset_json, f, indent=2)

converted_df = pd.DataFrame(converted_rows)
converted_df.to_csv(OUTPUT_ROOT / '02_13b_nnunet_external_ich_conversion_manifest.csv', index=False)

print('Dataset dir:', DATASET_DIR)
print('imagesTr:', len(list(imagesTr.glob('*.nii.gz'))))
print('labelsTr:', len(list(labelsTr.glob('*.nii.gz'))))
print('imagesTs:', len(list(imagesTs.glob('*.nii.gz'))))
print('labelsTs:', len(list(labelsTs.glob('*.nii.gz'))))
display(converted_df.groupby(['dataset', 'split']).agg(cases=('case_id', 'count'), ich_voxels=('ich_positive_voxels', 'sum')).reset_index())


Dataset dir: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_raw\Dataset123_ExternalICHTeacherImproved
imagesTr: 128
labelsTr: 128
imagesTs: 23
labelsTs: 23


,dataset,split,cases,ich_voxels
0,Instance2022,test,15,360074
1,Instance2022,train,70,1483441
2,Instance2022,val,15,772439
3,Seg-CQ500,test,8,1332203
4,Seg-CQ500,train,35,7664743
5,Seg-CQ500,val,8,637720


## 5. Helper gọi entrypoint nnU-Net

Dùng entrypoint cho planning/evaluation. Prediction dùng API trực tiếp để tránh lỗi PyTorch thread trong Jupyter.

In [8]:
def call_entrypoint(entrypoint_func, argv):
    old_argv = sys.argv[:]
    sys.argv = list(argv)
    print('>>>', ' '.join(map(str, sys.argv)))
    t0 = time.time()
    try:
        return entrypoint_func()
    finally:
        sys.argv = old_argv
        print(f'Done in {(time.time() - t0) / 60:.1f} min')


## 6. Plan and preprocess

`nnUNet_n_proc_DA` phải >= 1 khi planning vì planner gọi `torch.set_num_threads`. Training sẽ chuyển về 0 để tránh worker crash trên Windows/Jupyter.

In [9]:
CONFIGURATION = '3d_fullres'  # alternatives: '2d', '3d_fullres'
RUN_PLAN_PREPROCESS = True

if RUN_PLAN_PREPROCESS:
    from nnunetv2.experiment_planning.plan_and_preprocess_entrypoints import plan_and_preprocess_entry
    old_n_proc_da = os.environ.get('nnUNet_n_proc_DA')
    os.environ['nnUNet_n_proc_DA'] = '1'
    try:
        call_entrypoint(plan_and_preprocess_entry, [
            'nnUNetv2_plan_and_preprocess',
            '-d', str(DATASET_ID),
            '-c', CONFIGURATION,
            '--verify_dataset_integrity',
            '-npfp', '2',
            '-np', '1',
        ])
    finally:
        os.environ['nnUNet_n_proc_DA'] = old_n_proc_da if old_n_proc_da is not None else '0'
else:
    print('Skipped planning/preprocessing.')


>>> nnUNetv2_plan_and_preprocess -d 123 -c 3d_fullres --verify_dataset_integrity -npfp 2 -np 1
Fingerprint extraction...
Dataset123_ExternalICHTeacherImproved
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer

####################
verify_dataset_integrity Done. 
If you didn't see any error messages then your dataset is most likely OK!
####################

Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer


Extracting dataset fingerprint: 100%|██████████| 128/128 [00:26<00:00,  4.79it/s]


Experiment planning...

############################
INFO: You are using the old nnU-Net default planner. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Attempting to find 3d_lowres config. 
Current spacing: [0.6451411  0.48203999 0.48203999]. 
Current patch size: (80, 192, 160). 
Current median shape: [232.03883495 497.08737864 497.08737864]
Attempting to find 3d_lowres config. 
Current spacing: [0.66449533 0.49650119 0.49650119]. 
Current patch size: (80, 192, 160). 
Current median shape: [225.28042228 482.60910548 482.60910548]
Attempting to find 3d_lowres config. 
Current spacing: [0.68443019 0.51139623 0.51139623]. 
Current patch size: (80, 192, 160). 
Current median shape: [218.71885659 468.55252959 468.55252959]
Attempting to find 3d_lowres config. 
Current spacing: [0.7049631  0.52673812 0.52673812]. 
Current patch size: (80, 19

Preprocessing cases: 100%|██████████| 128/128 [22:58<00:00, 10.77s/it]


Done in 23.9 min


## 7. Ghi manual split fold 0

nnU-Net train trên `imagesTr/labelsTr`, gồm train + val. File `splits_final.json` ép fold 0 theo split mình đã tạo.

In [10]:
preprocessed_dataset_dir = NNUNET_PREPROCESSED / DATASET_NAME
split_json_path = preprocessed_dataset_dir / 'splits_final.json'

train_ids = manifest_df.loc[manifest_df['split'] == 'train', 'case_id'].astype(str).tolist()
val_ids = manifest_df.loc[manifest_df['split'] == 'val', 'case_id'].astype(str).tolist()
test_ids = manifest_df.loc[manifest_df['split'] == 'test', 'case_id'].astype(str).tolist()

manual_splits = [{'train': train_ids, 'val': val_ids}]

if preprocessed_dataset_dir.exists():
    with open(split_json_path, 'w', encoding='utf-8') as f:
        json.dump(manual_splits, f, indent=2)
    print('Wrote:', split_json_path)
    print('train:', len(train_ids), 'val:', len(val_ids), 'test:', len(test_ids))
else:
    print('Preprocessed folder not found yet:', preprocessed_dataset_dir)


Wrote: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_preprocessed\Dataset123_ExternalICHTeacherImproved\splits_final.json
train: 105 val: 23 test: 23


## 8. Train ICH teacher

Default dùng `nnUNetTrainer_250epochs`: đủ là một run thật nhẹ hơn default 1000 epoch. Nếu muốn smoke test, đổi `TRAINER = 'nnUNetTrainer_5epochs'`.

In [11]:
RUN_TRAIN = True
FOLD = 0
TRAINER = 'nnUNetTrainer_250epochs'
PLANS = 'nnUNetPlans'
EXPORT_VAL_NPZ = True
CONTINUE_TRAINING = True

if RUN_TRAIN:
    import torch
    from nnunetv2.run.run_training import run_training
    os.environ['nnUNet_n_proc_DA'] = '0'
    os.environ['nnUNet_compile'] = 'False'
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Training device:', device)
    print('nnUNet_n_proc_DA:', os.environ.get('nnUNet_n_proc_DA'))
    run_training(
        str(DATASET_ID),
        CONFIGURATION,
        str(FOLD),
        trainer_class_name=TRAINER,
        plans_identifier=PLANS,
        export_validation_probabilities=EXPORT_VAL_NPZ,
        continue_training=CONTINUE_TRAINING,
        device=device,
    )
else:
    print('Skipped training.')


Training device: cuda
nnUNet_n_proc_DA: 0

############################
INFO: You are using the old nnU-Net default plans. We have updated our recommendations. Please consider using those instead! Read more here: https://github.com/MIC-DKFZ/nnUNet/blob/master/documentation/resenc_presets.md
############################

Using device: cuda:0

#######################################################################
Please cite the following paper when using nnU-Net:
Isensee, F., Jaeger, P. F., Kohl, S. A., Petersen, J., & Maier-Hein, K. H. (2021). nnU-Net: a self-configuring method for deep learning-based biomedical image segmentation. Nature methods, 18(2), 203-211.
#######################################################################

2026-06-18 10:22:36.841768: do_dummy_2d_data_aug: False
2026-06-18 10:22:36.842768: Using splits from existing split file: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_preprocessed\Dataset123

## 9. Predict external test set + nnU-Net basic evaluation

Prediction dùng API `nnUNetPredictor`, không dùng command entrypoint, để tránh lỗi `torch.set_num_interop_threads` trong cùng Jupyter kernel sau training.

In [12]:
RUN_PREDICT = True
RUN_EVALUATE = True
DISABLE_TTA = True
SAVE_PROBABILITIES = True
CHECKPOINT = 'auto'  # auto: best -> final -> latest
PERFORM_EVERYTHING_ON_DEVICE = False

MODEL_DIR = NNUNET_RESULTS / DATASET_NAME / f'{TRAINER}__{PLANS}__{CONFIGURATION}'
FOLD_DIR = MODEL_DIR / f'fold_{FOLD}'

def resolve_checkpoint_name(checkpoint_setting='auto'):
    if checkpoint_setting != 'auto':
        ckpt = FOLD_DIR / checkpoint_setting
        if not ckpt.exists():
            available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
            raise FileNotFoundError(f'Missing checkpoint: {ckpt}. Available: {available}')
        return checkpoint_setting
    for name in ['checkpoint_best.pth', 'checkpoint_final.pth', 'checkpoint_latest.pth']:
        if (FOLD_DIR / name).exists():
            return name
    available = sorted(p.name for p in FOLD_DIR.glob('checkpoint*.pth')) if FOLD_DIR.exists() else []
    raise FileNotFoundError(f'No checkpoint found in {FOLD_DIR}. Available: {available}')

RESOLVED_CHECKPOINT = resolve_checkpoint_name(CHECKPOINT)
print('Model dir        :', MODEL_DIR)
print('Using checkpoint :', RESOLVED_CHECKPOINT)
print('Save probabilities:', SAVE_PROBABILITIES)

PRED_DIR = OUTPUT_ROOT / f'predictions_{CONFIGURATION}_{TRAINER}_fold{FOLD}_{RESOLVED_CHECKPOINT.replace(".pth", "")}'
SUMMARY_JSON = PRED_DIR / 'summary.json'

if RUN_PREDICT:
    import torch
    from nnunetv2.inference.predict_from_raw_data import nnUNetPredictor
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    print('Predict device:', device)
    predictor = nnUNetPredictor(
        tile_step_size=0.5,
        use_gaussian=True,
        use_mirroring=not DISABLE_TTA,
        perform_everything_on_device=PERFORM_EVERYTHING_ON_DEVICE,
        device=device,
        verbose=False,
        verbose_preprocessing=False,
        allow_tqdm=True,
    )
    predictor.initialize_from_trained_model_folder(
        str(MODEL_DIR),
        use_folds=(FOLD,),
        checkpoint_name=RESOLVED_CHECKPOINT,
    )
    predictor.predict_from_files(
        str(imagesTs),
        str(PRED_DIR),
        save_probabilities=SAVE_PROBABILITIES,
        overwrite=True,
        num_processes_preprocessing=1,
        num_processes_segmentation_export=1,
        folder_with_segs_from_prev_stage=None,
        num_parts=1,
        part_id=0,
    )
else:
    print('Skipped prediction.')

if RUN_EVALUATE:
    from nnunetv2.evaluation.evaluate_predictions import evaluate_folder_entry_point
    plans_file = NNUNET_PREPROCESSED / DATASET_NAME / f'{PLANS}.json'
    dataset_json_file = DATASET_DIR / 'dataset.json'
    call_entrypoint(evaluate_folder_entry_point, [
        'nnUNetv2_evaluate_folder',
        str(labelsTs),
        str(PRED_DIR),
        '-djfile', str(dataset_json_file),
        '-pfile', str(plans_file),
        '-o', str(SUMMARY_JSON),
        '-np', '2',
    ])
else:
    print('Skipped nnU-Net evaluation.')

print('Prediction dir:', PRED_DIR)
print('Summary json  :', SUMMARY_JSON)


Model dir        : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_results\Dataset123_ExternalICHTeacherImproved\nnUNetTrainer_250epochs__nnUNetPlans__3d_fullres
Using checkpoint : checkpoint_best.pth
Save probabilities: True
Predict device: cuda
There are 23 cases in the source folder
I am process 0 out of 1 (max process ID is 0, we start counting with 0!)
There are 23 cases that I would like to predict

Predicting inst2022_003:
perform_everything_on_device: False


100%|██████████| 180/180 [00:15<00:00, 11.52it/s]


sending off prediction to background worker for resampling and export
done with inst2022_003

Predicting inst2022_019:
perform_everything_on_device: False


100%|██████████| 180/180 [00:15<00:00, 11.68it/s]


sending off prediction to background worker for resampling and export
done with inst2022_019

Predicting inst2022_022:
perform_everything_on_device: False


...

100%|██████████| 210/210 [00:17<00:00, 11.77it/s]


sending off prediction to background worker for resampling and export
done with segcq500_CQ500_CT_55

Predicting segcq500_CQ500_CT_80:
perform_everything_on_device: False


100%|██████████| 210/210 [00:17<00:00, 11.97it/s]


sending off prediction to background worker for resampling and export
done with segcq500_CQ500_CT_80
GPU prediction completed. Waiting for remaining segmentation exports to finish...


Segmentation export complete.
>>> nnUNetv2_evaluate_folder D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_raw\Dataset123_ExternalICHTeacherImproved\labelsTs D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best -djfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_raw\Dataset123_ExternalICHTeacherImproved\dataset.json -pfile D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\nnunet_workdir\nnUNet_preprocessed\Dataset123_ExternalICHTeacherImproved\nnUNetPlans.json -o D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\summary.json -np 2
Using <class 'nnunetv2.imageio.simpleitk_reader_writer.SimpleITKIO'> reader/writer
Done in 0.1 m

## 9b. Export teacher probability maps

nnU-Net can save compressed probability arrays (`.npz`). This cell converts the ICH class probability into NIfTI maps aligned with the prediction geometry, and also writes thresholded binary masks.


In [13]:
RUN_EXPORT_TEACHER_PROBABILITIES = True

TEACHER_PROB_DIR = PRED_DIR / 'probability_maps'
TEACHER_BINARY_DIR = PRED_DIR / 'binary_masks'
TEACHER_PROB_DIR.mkdir(parents=True, exist_ok=True)
TEACHER_BINARY_DIR.mkdir(parents=True, exist_ok=True)

def load_nnunet_probability(npz_path: Path):
    with np.load(npz_path) as data:
        key = 'probabilities' if 'probabilities' in data.files else data.files[0]
        arr = np.asarray(data[key])
    if arr.ndim == 4:
        return arr[1] if arr.shape[0] > 1 else arr[0]
    if arr.ndim == 3:
        return arr
    raise ValueError(f'Unexpected probability shape {arr.shape}: {npz_path}')

def write_array_like_reference(arr, reference_path: Path, out_path: Path, dtype=np.float32):
    ref = sitk.ReadImage(str(reference_path))
    out = sitk.GetImageFromArray(np.asarray(arr).astype(dtype))
    if out.GetSize() != ref.GetSize():
        raise ValueError(f'Size mismatch for {out_path}: {out.GetSize()} vs ref {ref.GetSize()}')
    out.CopyInformation(ref)
    sitk.WriteImage(out, str(out_path))

if RUN_EXPORT_TEACHER_PROBABILITIES:
    export_rows = []
    for row in converted_df[converted_df['split'] == 'test'].itertuples(index=False):
        case_id = str(row.case_id)
        npz_path = PRED_DIR / f'{case_id}.npz'
        pred_path = PRED_DIR / f'{case_id}.nii.gz'
        if npz_path.exists():
            prob = load_nnunet_probability(npz_path)
            status = 'probability_npz'
        elif pred_path.exists():
            prob = (sitk.GetArrayFromImage(sitk.ReadImage(str(pred_path))) > 0).astype(np.float32)
            status = 'fallback_binary_prediction'
        else:
            print('Missing prediction/probability:', case_id)
            continue
        prob = np.clip(prob.astype(np.float32), 0.0, 1.0)
        binary = (prob >= TEACHER_PROB_THRESHOLD).astype(np.uint8)
        prob_path = TEACHER_PROB_DIR / f'{case_id}.nii.gz'
        binary_path = TEACHER_BINARY_DIR / f'{case_id}.nii.gz'
        ref_path = pred_path if pred_path.exists() else labelsTs / f'{case_id}.nii.gz'
        write_array_like_reference(prob, ref_path, prob_path, dtype=np.float32)
        write_array_like_reference(binary, ref_path, binary_path, dtype=np.uint8)
        export_rows.append({
            'case_id': case_id,
            'source': status,
            'probability_path': str(prob_path),
            'binary_path': str(binary_path),
            'prob_min': float(prob.min()),
            'prob_max': float(prob.max()),
            'prob_mean': float(prob.mean()),
            'binary_voxels': int(binary.sum()),
        })
    teacher_prob_export_df = pd.DataFrame(export_rows)
    export_csv = OUTPUT_ROOT / '02_13b_teacher_probability_export_manifest.csv'
    teacher_prob_export_df.to_csv(export_csv, index=False)
    display(teacher_prob_export_df.head())
    print('Saved probability maps:', TEACHER_PROB_DIR)
    print('Saved binary masks    :', TEACHER_BINARY_DIR)
    print('Saved manifest        :', export_csv)
else:
    print('Skipped teacher probability export.')


,case_id,source,probability_path,binary_path,prob_min,prob_max,prob_mean,binary_voxels
0,segcq500_CQ500_CT_137,probability_npz,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,1.106582e-15,0.999962,0.000454,28184
1,segcq500_CQ500_CT_139,probability_npz,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,8.028132e-20,0.999990,0.000909,58923
2,segcq500_CQ500_CT_15,probability_npz,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,3.819682e-24,0.999994,0.001683,95528
3,segcq500_CQ500_CT_210,probability_npz,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,0.000000e+00,0.999998,0.005078,426227
4,segcq500_CQ500_CT_35,probability_npz,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_1...,7.351556e-21,1.000000,0.000061,2402


Saved probability maps: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\probability_maps
Saved binary masks    : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\predictions_3d_fullres_nnUNetTrainer_250epochs_fold0_checkpoint_best\binary_masks
Saved manifest        : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\02_13b_teacher_probability_export_manifest.csv


## 10. Full metrics cho ICH teacher

nnU-Net summary chỉ có Dice/IoU/TP/FP/FN. Cell này bổ sung các metric mình cần cho báo cáo:

- Dice, Jaccard, precision, recall
- HD95, ASSD, NSD@1mm
- RVD, volume MAE ml, detection rate

In [14]:
RUN_FULL_METRICS = True

def load_sitk_mask(path: Path):
    img = sitk.ReadImage(str(path))
    arr = sitk.GetArrayFromImage(img)
    return img, (arr > 0)

def surface_mask(mask):
    from scipy import ndimage as ndi
    if not mask.any():
        return mask.astype(bool)
    eroded = ndi.binary_erosion(mask, structure=np.ones((3, 3, 3), dtype=bool), border_value=0)
    return np.logical_xor(mask, eroded)

def distance_metrics(pred, gt, spacing_zyx, tolerance_mm=1.0):
    from scipy import ndimage as ndi
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    if not pred.any() and not gt.any():
        return {'hd95_mm': np.nan, 'assd_mm': np.nan, 'nsd_1mm': np.nan}
    if not pred.any() or not gt.any():
        return {'hd95_mm': np.inf, 'assd_mm': np.inf, 'nsd_1mm': 0.0}
    surf_pred = surface_mask(pred)
    surf_gt = surface_mask(gt)
    if not surf_pred.any() or not surf_gt.any():
        return {'hd95_mm': np.inf, 'assd_mm': np.inf, 'nsd_1mm': 0.0}
    dt_to_gt = ndi.distance_transform_edt(~surf_gt, sampling=spacing_zyx)
    dt_to_pred = ndi.distance_transform_edt(~surf_pred, sampling=spacing_zyx)
    d_pred_gt = dt_to_gt[surf_pred]
    d_gt_pred = dt_to_pred[surf_gt]
    d_all = np.concatenate([d_pred_gt, d_gt_pred])
    return {
        'hd95_mm': float(np.percentile(d_all, 95)),
        'assd_mm': float(d_all.mean()),
        'nsd_1mm': float((d_all <= tolerance_mm).mean()),
    }

def binary_case_metrics(pred, gt, spacing_xyz):
    pred = pred.astype(bool)
    gt = gt.astype(bool)
    tp = int(np.logical_and(pred, gt).sum())
    fp = int(np.logical_and(pred, ~gt).sum())
    fn = int(np.logical_and(~pred, gt).sum())
    dice = np.nan if (2 * tp + fp + fn) == 0 else 2 * tp / (2 * tp + fp + fn)
    jaccard = np.nan if (tp + fp + fn) == 0 else tp / (tp + fp + fn)
    precision = np.nan if (tp + fp) == 0 else tp / (tp + fp)
    recall = np.nan if (tp + fn) == 0 else tp / (tp + fn)
    voxel_ml = float(np.prod(spacing_xyz) / 1000.0)
    pred_ml = float(pred.sum() * voxel_ml)
    gt_ml = float(gt.sum() * voxel_ml)
    rvd = np.nan if gt_ml == 0 else (pred_ml - gt_ml) / gt_ml
    spacing_zyx = tuple(float(x) for x in spacing_xyz[::-1])
    dist = distance_metrics(pred, gt, spacing_zyx, tolerance_mm=1.0)
    return {
        'dice': dice,
        'jaccard': jaccard,
        'precision': precision,
        'recall': recall,
        **dist,
        'rvd': rvd,
        'pred_volume_ml': pred_ml,
        'gt_volume_ml': gt_ml,
        'volume_abs_error_ml': abs(pred_ml - gt_ml),
        'detected': bool(pred.any()) if gt.any() else np.nan,
        'gt_positive': bool(gt.any()),
        'pred_positive': bool(pred.any()),
    }

if RUN_FULL_METRICS:
    metric_rows = []
    for row in converted_df[converted_df['split'] == 'test'].itertuples(index=False):
        case_id = str(row.case_id)
        gt_path = labelsTs / f'{case_id}.nii.gz'
        pred_path = PRED_DIR / f'{case_id}.nii.gz'
        if not pred_path.exists():
            print('Missing prediction:', pred_path)
            continue
        gt_img, gt = load_sitk_mask(gt_path)
        _, pred = load_sitk_mask(pred_path)
        m = binary_case_metrics(pred, gt, np.array(gt_img.GetSpacing(), dtype=float))
        metric_rows.append({'case_id': case_id, 'dataset': row.dataset, **m})
    metrics_df = pd.DataFrame(metric_rows)
    metrics_path = OUTPUT_ROOT / '02_13_external_ich_teacher_full_metrics_per_case.csv'
    summary_path = OUTPUT_ROOT / '02_13_external_ich_teacher_full_metrics_summary.csv'
    metrics_df.to_csv(metrics_path, index=False)
    summary_rows = []
    metric_cols = ['dice', 'jaccard', 'precision', 'recall', 'hd95_mm', 'assd_mm', 'nsd_1mm', 'rvd', 'volume_abs_error_ml']
    for group_name, group in [('all', metrics_df)] + [(d, g) for d, g in metrics_df.groupby('dataset')]:
        for col in metric_cols:
            values = pd.to_numeric(group[col], errors='coerce').replace([np.inf, -np.inf], np.nan)
            summary_rows.append({
                'group': group_name,
                'metric': col,
                'mean': float(values.mean()) if len(values) else np.nan,
                'median': float(values.median()) if len(values) else np.nan,
                'std': float(values.std()) if len(values) else np.nan,
                'n': int(values.notna().sum()),
            })
        summary_rows.append({
            'group': group_name,
            'metric': 'detection_rate',
            'mean': float(pd.to_numeric(group.loc[group['gt_positive'], 'detected'], errors='coerce').mean()),
            'median': np.nan,
            'std': np.nan,
            'n': int(group['gt_positive'].sum()),
        })
    summary_df = pd.DataFrame(summary_rows)
    summary_df.to_csv(summary_path, index=False)
    print('Saved per-case metrics:', metrics_path)
    print('Saved summary metrics :', summary_path)
    display(summary_df.pivot_table(index='metric', columns='group', values='mean'))
else:
    print('Skipped full metrics.')


Saved per-case metrics: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\02_13_external_ich_teacher_full_metrics_per_case.csv
Saved summary metrics : D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\02_13_external_ich_teacher_full_metrics_summary.csv


group,Instance2022,Seg-CQ500,all
metric,,,
assd_mm,3.287051,6.932725,4.555112
detection_rate,1.000000,1.000000,1.000000
dice,0.780521,0.766891,0.775780
hd95_mm,21.429379,39.671494,27.774463
jaccard,0.676123,0.642423,0.664401
nsd_1mm,0.792238,0.701191,0.760570
precision,0.814957,0.802809,0.810731
recall,0.771208,0.778840,0.773863
rvd,-0.066489,-0.001652,-0.043937


## 11. QC visualization external teacher

Mục tiêu là xem nhanh teacher có học đúng vùng máu/lesion không trước khi dùng nó predict trên PHE-SICH.

In [15]:
RUN_QC_FIGURES = True
N_QC_PER_BUCKET = 3
QC_DIR = OUTPUT_ROOT / 'figures_external_ich_teacher_qc'
QC_DIR.mkdir(parents=True, exist_ok=True)

CT_WINDOWS = {
    'brain': (0, 80),
    'blood': (-50, 150),
    'wide': (-100, 300),
}

def window_ct(arr, low=-50, high=150):
    arr = np.asarray(arr, dtype=np.float32)
    arr = np.clip(arr, low, high)
    return (arr - low) / max(high - low, 1e-6)

def largest_slice(mask):
    if not mask.any():
        return mask.shape[0] // 2
    areas = mask.reshape(mask.shape[0], -1).sum(axis=1)
    return int(np.argmax(areas))

def select_teacher_qc_cases(metrics_df, n_each=3):
    buckets = []
    buckets.append(metrics_df.sort_values('dice', na_position='last').head(n_each).assign(qc_reason='low_dice'))
    buckets.append(metrics_df.sort_values('dice', ascending=False, na_position='last').head(n_each).assign(qc_reason='high_dice'))
    buckets.append(metrics_df.sort_values('pred_volume_ml', na_position='last').head(n_each).assign(qc_reason='tiny_prediction'))
    buckets.append(metrics_df.sort_values('pred_volume_ml', ascending=False, na_position='last').head(n_each).assign(qc_reason='large_prediction'))
    out = pd.concat(buckets, ignore_index=True)
    return out.drop_duplicates('case_id', keep='first')

if RUN_QC_FIGURES:
    import matplotlib.pyplot as plt
    if 'metrics_df' not in globals():
        print('Run full metrics first to select QC cases.')
    else:
        qc_df = select_teacher_qc_cases(metrics_df, N_QC_PER_BUCKET)
        qc_df.to_csv(OUTPUT_ROOT / '02_13b_teacher_qc_selected_cases.csv', index=False)
        for row in qc_df.itertuples(index=False):
            case_id = str(row.case_id)
            image_path = imagesTs / f'{case_id}_0000.nii.gz'
            gt_path = labelsTs / f'{case_id}.nii.gz'
            pred_path = PRED_DIR / f'{case_id}.nii.gz'
            prob_path = TEACHER_PROB_DIR / f'{case_id}.nii.gz'
            img = sitk.GetArrayFromImage(sitk.ReadImage(str(image_path)))
            _, gt = load_sitk_mask(gt_path)
            _, pred = load_sitk_mask(pred_path)
            prob = sitk.GetArrayFromImage(sitk.ReadImage(str(prob_path))) if prob_path.exists() else pred.astype(np.float32)
            z = largest_slice(gt | pred)
            ct_brain = window_ct(img[z], *CT_WINDOWS['brain'])
            ct_blood = window_ct(img[z], *CT_WINDOWS['blood'])
            ct_wide = window_ct(img[z], *CT_WINDOWS['wide'])
            fig, axes = plt.subplots(1, 5, figsize=(18, 4))
            axes[0].imshow(ct_brain, cmap='gray')
            axes[0].set_title(f'{case_id}\nbrain window\n{row.qc_reason}')
            axes[1].imshow(ct_blood, cmap='gray')
            axes[1].imshow(np.ma.masked_where(~gt[z], gt[z]), cmap='Greens', alpha=0.55)
            axes[1].set_title('GT ICH')
            axes[2].imshow(ct_blood, cmap='gray')
            axes[2].imshow(np.ma.masked_where(~pred[z], pred[z]), cmap='Reds', alpha=0.55)
            axes[2].set_title('teacher binary')
            axes[3].imshow(ct_wide, cmap='gray')
            axes[3].imshow(prob[z], cmap='magma', alpha=0.60, vmin=0, vmax=1)
            axes[3].set_title('teacher probability')
            axes[4].imshow(ct_blood, cmap='gray')
            axes[4].imshow(np.ma.masked_where(~gt[z], gt[z]), cmap='Greens', alpha=0.45)
            axes[4].imshow(np.ma.masked_where(~pred[z], pred[z]), cmap='Reds', alpha=0.45)
            axes[4].set_title(f'overlay dice={row.dice:.3f}')
            for ax in axes:
                ax.axis('off')
            fig.tight_layout()
            out = QC_DIR / f'{case_id}_{row.qc_reason}_qc.png'
            fig.savefig(out, dpi=160, bbox_inches='tight')
            plt.close(fig)
        print('Saved QC figures:', QC_DIR)
else:
    print('Skipped QC figures.')


Saved QC figures: D:\Thuy_Loi\Nam_3\CT_xuathuyetnao\outputs_02_13b_nnunet_ich_teacher_external_improved\figures_external_ich_teacher_qc


## Kết luận stage 02_13

Nếu teacher ICH đạt chất lượng chấp nhận được trên external test và QC overlay ổn, bước tiếp theo là tạo notebook:

```text
02_14_generate_ich_teacher_prior_phe_sich.ipynb
```

Notebook đó sẽ dùng checkpoint teacher này để predict ICH trên toàn bộ PHE-SICH, lưu `ICH probability/mask`, rồi 3DFF-Net 2.5D sẽ dùng nó như prior input hoặc pseudo-label phụ.

Điểm cần ghi khi báo cáo: ICH teacher được học từ Seg-CQ500 + Instance2022; PHE final vẫn đánh giá trên PHE-SICH manual PHE mask.